In [25]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os
import numpy as np

In [26]:
# Create a DataFrame from your provided data
data = {
    "Learning Rate": [0.1, 0.01, 0.001, 0.0001, 0.00001],
    "ACC": [0.7658, 0.9130, 0.9790, 0.8627, 0.8078],
    "AUC": [0.9489, 0.9883, 0.9857, 0.9729, 0.9415],
    "PRE": [0.8352, 0.9102, 0.9066, 0.8555, 0.7977],
    "SP": [0.7465, 0.9062, 0.9029, 0.8524, 0.7924],
    "SN": [0.9228, 0.9712, 0.9040, 0.9540, 0.9351],
    "F1": [0.7530, 0.9076, 0.8795, 0.8515, 0.7897],
    "MCC": [0.7070, 0.8835, 0.9790, 0.8168, 0.7444]
}

In [27]:
df = pd.DataFrame(data)

In [28]:
# Reset matplotlib settings to default
plt.rcParams.update(plt.rcParamsDefault)

# Set Times New Roman font with fallback
plt.rcParams['font.family'] = ['Times New Roman', 'serif']
plt.rcParams['mathtext.fontset'] = 'stix'

# Set global plot style with large font sizes
plt.rcParams['font.size'] = 32
plt.rcParams['axes.labelsize'] = 36
plt.rcParams['axes.titlesize'] = 40
plt.rcParams['xtick.labelsize'] = 30
plt.rcParams['ytick.labelsize'] = 30
plt.rcParams['legend.fontsize'] = 22
plt.rcParams['figure.dpi'] = 1000
plt.rcParams['savefig.dpi'] = 1000
plt.rcParams['figure.facecolor'] = 'white'

In [29]:
learning_rate_palette = {
    0.1: '#2CA02C',     # Vivid Green (PCA)
    0.01: '#9467BD',    # Deep Purple (CHI2)
    0.001: '#1F77B4',   # Bright Blue (SHAP)
    0.0001: '#D62728',  # Bold Red (LASSO)
    0.00001: '#FF7F0E'  # Rich Orange (mRMR)
}

# Metric palette
metric_palettes = {
    'ACC': 'viridis',
    'AUC': 'plasma',
    'PRE': 'cividis',
    'SP': 'magma',
    'SN': 'inferno',
    'F1': 'copper',
    'MCC': 'Greens'
}

In [30]:
# Function to save figures in both PNG and PDF
def save_figure(fig, filename):
    output_folder = "Experiment_Output_Figures/Learning_Rate"
    if not os.path.exists(output_folder):
        os.makedirs(output_folder)
    
    png_path = os.path.join(output_folder, f"{filename}.png")
    pdf_path = os.path.join(output_folder, f"{filename}.pdf")
    
    fig.savefig(png_path, dpi=1000, bbox_inches='tight', facecolor='white', format='png')
    fig.savefig(pdf_path, dpi=1000, bbox_inches='tight', facecolor='white', format='pdf')
    
    print(f"Saved: {png_path} and {pdf_path}")
    plt.close(fig)

In [31]:
# Function to generate the violin plot visualization
def generate_violin_plot():
    metrics = ['ACC', 'AUC', 'PRE', 'SP', 'SN', 'F1', 'MCC']
    melted_df = pd.melt(df, id_vars=['Learning Rate'],
                       value_vars=metrics,
                       var_name='Metric',
                       value_name='Score')
    
    fig, ax = plt.subplots(figsize=(20, 14))
    sns.violinplot(x='Learning Rate', 
                   y='Score', 
                   hue='Learning Rate',
                   data=melted_df,
                   palette=learning_rate_palette, 
                   inner='box',
                   linewidth=2, 
                   ax=ax,
                   legend=False)
    
    ax.set_title('Distribution of Performance Scores by Learning Rate',
                fontsize=44, pad=30)
    ax.set_xlabel('Learning Rate', fontsize=40, labelpad=25)
    ax.set_ylabel('Score Distribution Across Metrics', fontsize=40, labelpad=25)
    ax.set_ylim(0.5, 1.1)  # Adjusted for your data range
    
    plt.tight_layout(pad=3.0)
    save_figure(fig, "learning_rate_violin_plot")

In [32]:
# Function to generate the timing comparison plot
def generate_timing_comparison():
    fig, ax = plt.subplots(figsize=(18, 10))
    sorted_df = df.sort_values(by='Training_Time', ascending=True)
    
    bars = ax.barh(sorted_df['Learning Rate'].astype(str), sorted_df['Training_Time'],
                  color=[learning_rate_palette[lr] for lr in sorted_df['Learning Rate']],
                  height=0.6)
    
    for i, learning_rate in enumerate(sorted_df['Learning Rate']):
        test_time = sorted_df[sorted_df['Learning Rate'] == learning_rate]['Testing_Time'].values[0]
        ax.text(10, i, f'Test: {test_time:.4f}s', ha='left', va='center',
                fontsize=22, color='black')
    
    ax.set_title('Training and Testing Time by Learning Rate', fontsize=40, pad=30)
    ax.set_xlabel('Training Time (seconds)', fontsize=36, labelpad=25)
    ax.set_ylabel('Learning Rate', fontsize=36, labelpad=25)
    
    max_time = sorted_df['Training_Time'].max()
    ax.set_xlim(0, max_time * 1.2)
    
    plt.tight_layout(pad=3.0)
    save_figure(fig, "learning_rate_timing_comparison")

In [33]:
metrics = ['ACC', 'AUC', 'PRE', 'SP', 'SN', 'F1', 'MCC']
melted_df = pd.melt(df, id_vars=['Learning Rate'],
                   value_vars=metrics,
                   var_name='Metric',
                   value_name='Score')

# Grouped bar chart
fig, ax = plt.subplots(figsize=(20, 14))
sns.barplot(x='Metric', y='Score', hue='Learning Rate',
            data=melted_df, palette=learning_rate_palette,
            ax=ax, edgecolor='none')

ax.set_title('Comparison of Learning Rates across Metrics', fontsize=44, pad=30)
ax.set_xlabel('Evaluation Metric', fontsize=40, labelpad=25)
ax.set_ylabel('Score', fontsize=40, labelpad=25)
ax.set_ylim(0.25, 1.01)  # Adjusted for your data range

ax.legend(title='Learning Rate', title_fontsize=24, fontsize=22,
          bbox_to_anchor=(1.05, 1), loc='upper left')

plt.tight_layout(pad=3.0)
save_figure(fig, "learning_rate_comparison_grouped_bar")

Saved: Experiment_Output_Figures/Learning_Rate/learning_rate_comparison_grouped_bar.png and Experiment_Output_Figures/Learning_Rate/learning_rate_comparison_grouped_bar.pdf


In [34]:
# Radar Chart
categories = metrics
N = len(categories)
angles = [n / float(N) * 2 * np.pi for n in range(N)]
angles += angles[:1]

fig, ax = plt.subplots(figsize=(16, 16), subplot_kw=dict(polar=True))
for i, learning_rate in enumerate(df['Learning Rate']):
    values = df.loc[i, metrics].values.tolist()
    values += values[:1]
    ax.plot(angles, values, linewidth=4, label=str(learning_rate),
            color=learning_rate_palette[learning_rate])
    ax.fill(angles, values, alpha=0.1, color=learning_rate_palette[learning_rate])

ax.set_ylim(0.25, 1.01)  # Adjusted for your data range
plt.xticks(angles[:-1], categories, fontsize=36)
ax.legend(loc='upper right', bbox_to_anchor=(1.3, 1.0), fontsize=22)
plt.title('Learning Rate Comparison (Radar Chart)',
          fontsize=44, pad=40, y=1.08)

plt.tight_layout(pad=3.0)
save_figure(fig, "learning_rate_radar_chart")

Saved: Experiment_Output_Figures/Learning_Rate/learning_rate_radar_chart.png and Experiment_Output_Figures/Learning_Rate/learning_rate_radar_chart.pdf


In [35]:
# Heatmap
heatmap_df = df[['Learning Rate'] + metrics].set_index('Learning Rate')
fig, ax = plt.subplots(figsize=(18, 12))

heatmap = sns.heatmap(heatmap_df, annot=True, fmt=".4f", cmap="YlGnBu",
                     linewidths=0.5, linecolor='white', annot_kws={'size': 26},
                     vmin=0.25, vmax=1.0)  # Adjusted for your data range

cbar = heatmap.collections[0].colorbar
cbar.set_label('Score', size=34)
cbar.ax.tick_params(labelsize=28)

ax.set_title('Learning Rate Performance Heatmap', fontsize=40, pad=30)
ax.set_xlabel('Evaluation Metrics', fontsize=36, labelpad=25)
ax.set_ylabel('Learning Rate', fontsize=36, labelpad=25)

plt.tight_layout(pad=3.0)
save_figure(fig, "learning_rate_heatmap")

Saved: Experiment_Output_Figures/Learning_Rate/learning_rate_heatmap.png and Experiment_Output_Figures/Learning_Rate/learning_rate_heatmap.pdf


In [36]:
generate_violin_plot()

Saved: Experiment_Output_Figures/Learning_Rate/learning_rate_violin_plot.png and Experiment_Output_Figures/Learning_Rate/learning_rate_violin_plot.pdf


In [37]:
# generate_timing_comparison()

In [38]:
print("All learning rate visualizations have been generated!")
print(f"Files are saved in the 'Learning_Rate' folder")

All learning rate visualizations have been generated!
Files are saved in the 'Learning_Rate' folder
